In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

c:\Swdtools\conda_envs\py311_agenticai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
model = "openai/gpt-4.1-mini"

In [4]:
llm = ChatOpenAI(
    model=model,
    api_key=token,
    base_url=endpoint
)

In [5]:
from langchain_community.tools.tavily_search import TavilySearchResults


In [6]:
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
tavily_tool = TavilySearchResults(tavily_api_key=TAVILY_API_KEY)

C:\Users\prayag sonar\AppData\Local\Temp\ipykernel_34796\1788901970.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(tavily_api_key=TAVILY_API_KEY)


In [7]:
tavily_tool.invoke("what the latest of india vs westindies match")

[{'title': 'India beat West Indies by an innings and 140 runs in Ahmedabad ...',
  'url': 'https://indianexpress.com/article/sports/cricket/ind-vs-wi-live-score-india-indies-1st-test-match-day-2-live-cricket-score-highlights-updates-narendra-modi-stadium-gujarat-10284409/',
  'content': "India, playing their first Test at home in 2025, are on course for victory in just three days. India have not lost a Test to West Indies at home since 1994, winning eight games and drawing two in the phase. The West Indies' last series victory over India in India came way back in 1983! There are very mimimal chances of that being repeated this year at least with India just needing five wickets to win.\n\n## IND vs WI LIVE Score, 1st Test Day 3: NO HOPE! [...] ## How the wobbled seam helped Mohammed Siraj run through West Indies on Day 1 at Ahmedabad\n\n India's Mohammed Siraj celebrates the dismissal of West Indies' Alick Athanaze on the first day of the first Test cricket match between India and West 

In [8]:
my_code = """
x=10
y=x+10
print(y)
"""

In [9]:
from langchain_experimental.utilities import PythonREPL

In [10]:
repl = PythonREPL()
repl.run(my_code)

Python REPL can execute arbitrary code. Use with caution.


'20\n'

In [11]:
from typing import Annotated
from langchain.tools import tool

@tool
def python_repl_tool(
    code:Annotated[str, "the python code to run"],):

    """Use this to execute python code and do match. if you want to see the ouput of a value,
    you should print it out with 'print()' function. This is visible to user.
    """

    try:
        result = repl.run(code)
    except BaseException as e:
        return f"failed to execute python code: {e}"

    result_str = f"Successfully executed python code: {code}. stderr and stdout is {result}"

    return result_str

In [12]:
python_repl_tool.invoke(my_code)

'Successfully executed python code: \nx=10\ny=x+10\nprint(y)\n. stderr and stdout is 20\n'

In [13]:
members=["researhcher","coder"]

In [14]:
options = members+["FINISH"]


In [15]:
options

['researhcher', 'coder', 'FINISH']

In [16]:
from typing import Literal, TypedDict


class Router(TypedDict):
    next: Literal["researhcher","coder","FINISH"]


In [27]:
from langgraph.graph import MessagesState, StateGraph, START, END
class State(MessagesState):
    next:str


In [18]:
system_prompt= """
    you are supervisor , tasked with manging conversion between following workers{members}
    give the following user request, respond with the worker to act next. Each worker will perform
    a task and respond with their results.
"""

In [20]:
from langgraph.types import Command

In [ ]:
from turtle import goto


def supervisor_agent(state:State)->Command[Literal['researcher,coder,FINISH']]:
    messages= [{"role":"system","content":system_prompt}]+ state['messages']

    llm_with_structured_output=llm.with_structured_output(Router)
    response = llm_with_structured_output.invoke(messages)

#this is my next worker agent
    goto = response["next"]

    print(goto)

    if goto == "FINISH":
        goto = END
    return Command(goto = goto, update={"next": goto})

: 

In [ ]:
def researcher_agent(state:State)->Command[Literal['supervisor']]:
    research_agent = create_react_agent(llm, tools=[search_tool], prompt="you are a researcher.")
    result = researcher_agent.invoke(state)

    return Command(
        update={
            "messages":[HumanMessage(content=result["messages"][-1].content, name="researcher")]
        },

    )

In [23]:
def coder_agent(state:State)->Command[Literal['supervisor']]:
    pass

In [ ]:
class StateGraph(State):
    next:str

In [28]:
graph = StateGraph(State)

In [29]:
graph.add_node("supervisor",supervisor_agent)

In [30]:
graph.add_node("researcher",researcher_agent)

In [31]:
graph.add_node("coder", coder_agent)

In [32]:
graph.add_edge(START,"supervisor")

In [33]:
app = graph.compile()

ValueError: Found edge ending at unknown node `researcher,coder,FINISH`

In [34]:
from IPython.display import display,Image

In [ ]:
display(Image(app.get_graph().draw_marmaid()))